# Drugi domaći zadatak iz Digitalne obrade signala
## Student: Vasilije Čarapić 2022/0218

In [ ]:
USE_WIDGETS = True
def importEssentialLibs(USE_WIDGETS): 
    import numpy as np
    if USE_WIDGETS:
        %matplotlib widget
    else:
        %matplotlib inline
    import matplotlib as mpl 
    mpl.rc('text', usetex = True)
    mpl.rc('font', family = 'serif', size = 18)
    import scipy.signal as signal 
    import matplotlib.pyplot as plt 
       
    return np,mpl,plt,signal

# DEO 1-REALIZACIJA IIR FILTRA 

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from  matplotlib import patches

def zplane(b, a, ax = None):
    """Plot the complex z-plane given a transfer function.
    """

    # get a figure/plot
    if ax == None:
        w, h = plt.figaspect(1)
        fig, ax = plt.subplots(1, figsize=(w, h))
        plt.subplots_adjust(left = 0.2, bottom = 0.2)

    # create the unit circle
    uc = patches.Circle((0,0), radius=1, fill=False, color='black', ls='dashed')
    ax.add_patch(uc)
        
    # Get the poles and zeros
    p = np.roots(a)
    z = np.roots(b)
    k = b[0]/a[0]
    
    ## Count same poles/zeros
    # Make poles and zeros dictionaries
    poleCounts = {}
    for i in range(len(p)):
        if p[i].imag < 0:
            key = f'%f-%fj' % (p[i].real, -p[i].imag)
        else:
            key = f'%f+%fj' % (p[i].real, p[i].imag)
        if key in poleCounts:
            poleCounts[key] += 1
        else:
            poleCounts[key] = 1
    zeroCounts = {}
    
    for i in range(len(z)):
        if z[i].imag < 0:
            key = f'%f-%fj' % (z[i].real, -z[i].imag)
        else:
            key = f'%f+%fj' % (z[i].real, z[i].imag)
        if key in zeroCounts:
            zeroCounts[key] += 1
        else:
            zeroCounts[key] = 1
    
    # Plot the poles and set marker properties
    poleKeys = list(poleCounts.keys())
    for i in range(len(poleKeys)):
        pole = complex(poleKeys[i])
        ti = plt.plot(pole.real, pole.imag, 'rx', ms=10)
        plt.setp( ti, markersize=10.0, markeredgewidth=3.0, markeredgecolor='r', markerfacecolor='r')
        # place number of the same poles near the marker
        if poleCounts[poleKeys[i]] > 1:
            textstr = f'%d' % (poleCounts[poleKeys[i]])
            ax.text(pole.real+0.05, pole.imag+0.05, textstr, color='r')

    # Plot the zeros and set marker properties
    zeroKeys = list(zeroCounts.keys())
    for i in range(len(zeroKeys)):
        zero = complex(zeroKeys[i])
        t2 = plt.plot(zero.real, zero.imag, 'go', ms=10)
        plt.setp( t2, markersize=12.0, markeredgewidth=3.0, markeredgecolor='g', markerfacecolor='None')
        # place number of the same zeros near the marker
        if zeroCounts[zeroKeys[i]] > 1:
            textstr = f'%d' % (zeroCounts[zeroKeys[i]])
            ax.text(zero.real+0.05, zero.imag+0.05, textstr, color='g')
    allVals = np.concatenate((abs(p.real), abs(p.imag), abs(z.real), abs(z.imag)));
    limVal = max(allVals)*1.1
    if limVal < 1.1:
        limVal = 1.1
    
    ax.plot([-limVal, limVal], [0, 0], 'k--')
    ax.plot([0, 0], [-limVal, limVal], 'k--')
    
    ax.set_xlim([-limVal, limVal])
    ax.set_ylim([-limVal, limVal])
    
    ax.set_xlabel('Real')
    ax.set_ylabel('Imaginary')
    
    return z, p, k, ax

## 1.1 Ucitavanje signala iz fajla i prikazivanje grafika

In [ ]:
np, mpl, plt, signal = importEssentialLibs(USE_WIDGETS)
from scipy.io import loadmat

#citanje signala
ecg_dict = loadmat("dz2_signali/ecg_corrupted.mat")

if "ecg_corrupted" in ecg_dict:
    ecg_array = ecg_dict["ecg_corrupted"]
    ecg_array = ecg_array[0, :]

fs = 360
Ts = 1 / fs
t = np.arange(len(ecg_array)) * Ts

fig, ax = plt.subplots(figsize=(10, 4))
fig.subplots_adjust(bottom=0.20)
ax.plot(t, ecg_array)
ax.set_ylabel("ECG signal")
ax.set_xlabel(r"$t$ [s]")
ax.grid(True)
plt.show()


## 1.2 Definicija funkcije za VF filtar

In [ ]:
import math
np,mpl,plt,signal = importEssentialLibs(True)

def baselineDriftFilter(fs, fa, fp, Aa, Ap):
    #1 nadje se Omega Wp=2 * np.pi*fp/fs, i Wa = 2*np.pi*fa/fs
    Wp = 2 * np.pi * fp/fs;
    Wa = 2 * np.pi * fa/fs;

    #2 predistorzija ucestanosti se uradi da bi se poklapale digitalna i analogna karakteristika
    wp = 2 * fs * np.tan(Wp / 2)
    wa = 2 * fs * np.tan(Wa / 2)

    #3 sad se nadje obicna karakteristika wpN jer je waN = 1 za cheb2Ap
    #wp/wa = waN/wpN -> wpN = (wa/wp)*waN
    waN = 1
    wpN = (wa / wp) * waN


    #4 sad se ubace formule da se nadje red filtra iz ovoga za cheb2ap
    D = (10 ** (0.1 * Aa) - 1) / (10 ** (0.1 * Ap) - 1)
    k = wpN/waN
    eq = np.arccosh(np.sqrt(D))/np.arccosh(1 / k)
    N = math.ceil(eq)

    z, p, k = signal.cheb2ap(N,Aa)
    epsilon = np.sqrt(np.power(10, 0.1 * Ap) - 1)

    #5 nadjemo an i bn nakon sto smo uradili aproksimaciju chebisevljevu
    an = np.array(np.poly(p))
    bn = k*np.array(np.poly(z))
    if an.shape == ():
        an = np.array([an])
    if bn.shape == ():
        bn = np.array([bn])

    #6 sad treba da se uradi b,a = signal.lp2hp() i nacrta krug
    b, a = signal.lp2hp(bn, an, wa)

    #7 sad se treba preci bilinearnom transformacijom( ne moze impulsno invarijantna za HF i NO filtre) u digitalni filtar
    [bd, ad] = signal.bilinear(b, a, fs)

    return [bd, ad]


## 1.3 Definicija NO filtra sa eliptickim normalizovanim prototipom

In [ ]:
def powerLineNoiseFilter(fs,fc,Aa,Ap):

    #1. izracunamo ucestanossti digitalnog filtra
    Fp1, Fp2 = (fc - 2) / fs, (fc + 2) / fs
    Fa1, Fa2  = (fc - 0.5) /fs, (fc + 0.5) / fs
    Wp1, Wp2 = 2 * np.pi * Fp1, 2 * np.pi * Fp2
    Wa1, Wa2 = 2 * np.pi *Fa1 , 2 * np.pi * Fa2

    #2. mora se uraditi predistorzija da bi se poklapale digitalna i analogna karakterstika filtra
    wp1, wp2 = 2 * fs * np.tan(Wp1 / 2), 2 * fs * np.tan(Wp2 / 2 )
    wa1, wa2 = 2 * fs * np.tan(Wa1 / 2), 2 * fs * np.tan(Wa2 / 2)
    w0 = np.sqrt(wp1 * wp2)
    wpN = 1
    B0 = (wp2 - wp1) * wpN

    #provera da li je uslov zadovoljen
    if wa2 < w0 ** 2 / wa1:
        print(wa1)
        wa1 = w0 ** 2 / wa2
        print(wa1)
    #3. mora se dobiti waN
    waN = B0 * wa1 / (w0 ** 2 - wa1**2)
    
    #4 sad se ispisu formule za filtar
    # Biram da projektujem sa eliptickom aproksimacijom jer se dobije najmanji red filtra
    D = (10**(0.1*Aa) - 1)/(10**(0.1*Ap) - 1)
    k = np.sqrt(1 - (wpN/waN)**2)
    q0 = 0.5*(1 - np.sqrt(k))/(1 + np.sqrt(k))
    q = q0 + 2*q0**5 + 15*q0**9 + 15*q0**13
    N = np.ceil((np.log10(16*D))/(np.log10(1/q)))

    z, p, k = signal.ellipap(N,Ap,Aa)

    #5 nadjimo koeficijente an, bn
    an = np.array(np.poly(p))
    bn = k * np.array(np.poly(z))

    if an.shape == ():
        an = np.array([an])
    if bn.shape == ():
        bn = np.array([bn])

    #5. sad treba da se uradi transformacija ucestanosti
    b, a = signal.lp2bs(bn, an, w0, B0)

    #6. bilinearnom transformacijom prelazimo u digitalni filtar
    [bd, ad] = signal.bilinear(b, a, fs)

    return [bd, ad]

## 1.4 Skiciranje nakon filtriranja signala VF filtrom

In [ ]:
h = baselineDriftFilter(360, 0.4, 1, 30, 0.5)
bd, ad= h[0], h[1]
filtered_signal = signal.lfilter(bd, ad, ecg_array)

t = np.arange(len(ecg_array)) / fs
fig, ax = plt.subplots(figsize = (10,4))
fig.subplots_adjust(bottom=0.20)
ax.plot(t, filtered_signal)
ax.set_ylabel("high-pass filtered signal")
ax.set_xlabel(r"$t$ [s]")
ax.grid(True)
plt.show()

## 1.5 Filtriranje signala sa NO filtrom nakon što je filtriran sa VF filtrom

In [ ]:
h = powerLineNoiseFilter(360, 60, 40, 0.5)
bd, ad= h[0], h[1]
end_signal = signal.lfilter(bd, ad, filtered_signal)

t = np.arange(len(filtered_signal)) / fs
fig, ax = plt.subplots(figsize = (10,4))
fig.subplots_adjust(bottom=0.20)
ax.plot(t, end_signal)
ax.set_ylabel("end product filtered signal")
ax.set_xlabel(r"$t$ [s]")
ax.grid(True)
plt.show()

## 1.6 Izračunavanje i crtanje amplitudskih karakteristika IIR filtra pomoću iir.design(...) metode 

In [ ]:
def IirHighPass(fs, fa, fp, Aa, Ap):
    # 1.izracunamo Wa,Wp
    Wa, Wp = 2 * np.pi * fa / fs, 2 * np.pi *fp / fs
    # 2. skaliramo ih sa pi i prosledimo u iirdesign
    WaScaled,WpScaled = Wa/np.pi, Wp/np.pi
    bd, ad = signal.iirdesign(WpScaled, WaScaled, Ap, Aa ,analog = False,ftype = "cheby2", output = "ba",fs = None)
    return [bd, ad]


In [ ]:
def IirBandStop(fs,fc,Aa,Ap):
    # 1. izracunamo ucestanossti digitalnog filtra
    Fp1, Fp2 = (fc - 2) / fs, (fc + 2) / fs
    Fa1, Fa2 = (fc - 0.5) / fs, (fc + 0.5) / fs
    
    Wp1, Wp2 = 2 * np.pi * Fp1, 2 * np.pi * Fp2
    Wa1, Wa2 = 2 * np.pi * Fa1, 2 * np.pi * Fa2

    # 2.skaliramo te izracunate ucestanosti
    wpVector = np.array([Wp1, Wp2]) / np.pi;
    waVector = np.array([Wa1, Wa2]) / np.pi;
    bd, ad = signal.iirdesign(wpVector, waVector,Ap, Aa, analog = False,ftype = "ellip",output = "ba", fs = None)

    #3. samo vratimo koeficijente filtra
    return [bd, ad]

In [ ]:
np, mpl, plt, signal = importEssentialLibs(True)
h = IirHighPass(360, 0.4, 1, 30, 0.5)
bd, ad= h[0], h[1]
fd, Hd = signal.freqz(bd, ad, worN = 5760,fs = fs) 
Hda = abs(Hd)
fs = 360 
fa = 0.4
fp = 1
Aa = 30
Ap = 0.5

fig, ax = plt.subplots(figsize = (10,4))
fig.subplots_adjust(bottom=0.20)
ax.plot(fd, 20*np.log10(Hda))  

rXlim = 25
ax.set_xlim([0, rXlim])

#crtanje gabarita 
# Stop band lim
ax.add_patch(mpl.patches.Polygon([[0, -Aa], [fa, -Aa], [fa, -Aa + 2], [0, -Aa + 2]], closed=True,
                                      fill=False, hatch='////', color='red'))
# Pass band lim
ax.add_patch(mpl.patches.Polygon([[fp, -Ap - 2], [rXlim, -Ap - 2], [rXlim, -Ap], [fp, -Ap]], closed=True,
                                      fill=False, hatch='////', color='red'))
ax.set_title("iir.design VF filtar")
ax.set_ylabel(r"$|H(e^{j\Omega})|[db]$")
ax.set_xlabel(r"$fd$ [Hz]")
ax.grid(True)
plt.show()


In [ ]:
np, mpl, plt, signal = importEssentialLibs(True)
h = IirBandStop(360, 60, 40, 0.5)
bd, ad = h[0], h[1]
fs = 360
Ts = 1/fs
fc = 60 
Aa = 40
Ap = 0.5

fp1, fp2 = (fc - 2) , (fc + 2)
fa1, fa2 = (fc - 0.5) , (fc + 0.5)

fd, Hd = signal.freqz(bd, ad, worN = 25000, fs = fs) 
Hda = abs(Hd)

fig, ax = plt.subplots(figsize = (10,4))
ax.set_xlim(50,70)
fig.subplots_adjust(bottom=0.20)
ax.plot(fd, 20*np.log10(Hda))
rXlim = 70

# prvi propusni opseg
ax.add_patch(mpl.patches.Polygon([[0, -Ap-2], [fp1, -Ap-2], [fp1, -Ap], [0, -Ap]],closed = True,
                  fill = False,hatch = "////",color = "red"  ))
# nepropusni opseg
ax.add_patch(mpl.patches.Polygon([[fa1, -Aa], [fa2, -Aa], [fa2, -Aa+2], [fa1, -Aa + 2]], closed=True,
                  fill=False, hatch="////", color="red"))
# drugi propusni opseg
ax.add_patch(mpl.patches.Polygon([[fp2, -Ap - 2], [rXlim, -Ap-2], [rXlim, -Ap], [fp2, -Ap]], closed=True,
                  fill=False, hatch="////", color="red"))


ax.set_title("iir.design bandstop filtar")
ax.set_ylabel(r"$|H(e^{j\Omega})|[db]$")
ax.set_xlabel(r"$fd$ [Hz]")
ax.grid(True)
plt.show()



## 1.7 Crtanje amplitudskih i faznih karakteristika filtara

### Amplitudska i fazna karakteristika za VF filtar 


In [ ]:
np, mpl, plt, signal = importEssentialLibs(True)
h = baselineDriftFilter(360, 0.4, 1, 30, 0.5)
bd, ad = h[0], h[1]
fd, Hd = signal.freqz(bd, ad, worN = 5760,fs = fs) 

fig = plt.figure(figsize=(12,5))
ax1 = fig.add_subplot(2,2,1)
ax1.set_title("Digitalni filtar")
zplane(bd, ad, ax1)

fd, Hd = signal.freqz(bd, ad, worN = 5760, fs = fs) 
Hda = abs(Hd)
fs = 360 
fa = 0.4
fp = 1
Aa = 30
Ap = 0.5

ax2 = fig.add_subplot(2,1,2)
fig.subplots_adjust(bottom=0.20,hspace = 0.5)
ax2.plot(fd, 20*np.log10(Hda))  

rXlim = 25
ax2.set_xlim([0, rXlim])

#crtanje gabarita

# nepropusni opseg
ax2.add_patch(mpl.patches.Polygon([[0, -Aa], [fa, -Aa], [fa, -Aa + 2], [0, -Aa + 2]], closed=True,
                                      fill=False, hatch='////', color='red'))
# propusni opseg
ax2.add_patch(mpl.patches.Polygon([[fp, -Ap - 2], [rXlim, -Ap - 2], [rXlim, -Ap], [fp, -Ap]], closed=True,
                                      fill=False, hatch='////', color='red'))
ax2.set_title("Amplitudska karakteristika")
ax2.set_ylabel(r"$|H(e^{j\Omega})|[db]$")
ax2.set_xlabel(r"$fd$ [Hz]")


ax3 = fig.add_subplot(2,2,2)
fig.subplots_adjust(wspace = 0.6, hspace=0.6)
ax3.plot(fd, np.unwrap(np.angle(Hd)))
ax3.set_title("Fazna karakteristika")
ax3.set_ylabel(r"$\angle H(e^{j\Omega})$ [rad]")
ax3.set_xlabel(r"$fd$ [Hz]")         

plt.show()

### Amplitudska i fazna karakteristika  za NO filtar 

In [ ]:
np, mpl, plt, signal = importEssentialLibs(True)
h = powerLineNoiseFilter(360, 60, 40, 0.5)
bd, ad = h[0], h[1]
fd, Hd = signal.freqz(bd, ad, worN = 5760,fs = fs) 

fig = plt.figure(figsize=(12,5))
ax1 = fig.add_subplot(2,2,1)
ax1.set_title("Digitalni filtar")
zplane(bd, ad, ax1)

fs = 360
Ts = 1/fs
fc = 60 
Aa = 40
Ap = 0.5

fp1 = (fc - 2) 
fp2 = (fc + 2)
fa1 = (fc - 0.5)
fa2 = (fc + 0.5)

Fa1 = fa1 / fs
Fa2 = fa2 / fs
Fp1 = fp1 / fs
Fp2 = fp2 / fs

Wa1 = 2 * np.pi * Fa1
Wa2 = 2 * np.pi * Fa2
Wp1 = 2 * np.pi * Fp1
Wp2 = 2 * np.pi * Fp2 

fd, Hd = signal.freqz(bd, ad, worN = 7200, fs = fs) 
Hda = abs(Hd)

ax2 = fig.add_subplot(2,1,2)
ax2.set_xlim(50,70)
fig.subplots_adjust(bottom=0.20,hspace=0.9)
ax2.plot(fd, 20*np.log10(Hda))
rXlim = 70

# prvi propusni opseg
ax2.add_patch(mpl.patches.Polygon([[0, -Ap-3], [fp1, -Ap-3], [fp1, -Ap], [0, -Ap]],closed = True,
                  fill = False,hatch = "////",color = "red"  ))
# nepropusni opseg
ax2.add_patch(mpl.patches.Polygon([[fa1, -Aa], [fa2, -Aa], [fa2, -Aa + 3], [fa1, -Aa + 3]], closed=True,
                  fill=False, hatch="////", color="red"))
# drugi propusni opseg
ax2.add_patch(mpl.patches.Polygon([[fp2, -Ap - 3], [rXlim, -Ap-3], [rXlim, -Ap], [fp2, -Ap]], closed=True,
                  fill=False, hatch="////", color="red"))


ax2.set_title("Amplitudska karakteristika")
ax2.set_ylabel(r"$|H(e^{j\Omega})|[db]$")
ax2.set_xlabel(r"$fd$ [Hz]")


ax3 = fig.add_subplot(2,2,2)
fig.subplots_adjust(wspace = 0.6)
ax3.plot(fd, np.unwrap(np.angle(Hd)))
ax3.set_title("Fazna karakteristika")
ax3.set_ylabel(r"$\angle H(e^{j\Omega})$ [rad]")
ax3.set_xlabel(r"$fd$ [Hz]")         

plt.show()

# DEO 2-REALIZACIJA FIR FILTRA  


## 2.1 Određivanje potrebnih FIR filtara koji su neophodni da se naprave kasnije 

In [ ]:
# ovo stoji ako se ne koristi ax.pcolormesh za prikazivanje spektrograma
def forceAspect(ax, aspect=1):
    im = ax.get_images()
    extent =  im[0].get_extent()
    ax.set_aspect(abs((extent[1]-extent[0])/(extent[3]-extent[2]))/aspect)

In [ ]:
np, mpl, plt, signal = importEssentialLibs(True)
import IPython
from scipy.io import wavfile
import math

fs,corrupted_signal =  wavfile.read("dz2_signali/sound_corrupted.wav")  
IPython.display.display(IPython.display.Audio(corrupted_signal,rate = fs)) 

t = np.arange(len(corrupted_signal)) / fs

fig,ax = plt.subplots(figsize = (11,4))
plt.subplots_adjust(bottom=0.20)
fMaxShow = 8000

Nwin = 3*1024;
window = signal.hamming(Nwin) 
NFFT = Nwin

f, t, Sxx  = signal.spectrogram(corrupted_signal, fs = fs,  window = window, noverlap = Nwin // 8, nfft = NFFT, return_onesided=True, 
                               scaling='spectrum', mode='complex')

fMaxIndex = NFFT*fMaxShow//fs

ax.pcolormesh(t, f[:fMaxIndex], 20*np.log10(abs(Sxx[:][:fMaxIndex])), shading = 'nearest', cmap = "magma")# try shading='gouraud')
ax.set_xlabel(r"$t[s]$")
ax.set_ylabel(r"$f$ [Hz]")
ax.set_title("Spektrogram ucitanog signala")
#print(fs)

Potrebno je uzeti dva NO filtra da bi se uklonie komponente šuma koje možemo primetiti da su dominantno izražene na učestanostima 1500 Hz i 2700 Hz. Razlog zbog koga je potreban nepropusnik opsega je taj sto želimo da uklonimo šum iz određenog opsega, to jest želimo da nam je opseg nepropusan. Takodje imamo štetne komponente na frekvencijama 6500+ zbog čega se dolazi do zaključka da se te visokofrekventne komponente trebaju potisnuti sa NF filtrom jer želimo da propustimo niskofrekventne komponente a da visokofrekventne filtriramo.


## 2.3 FIltriranje signala i prikazivanje spektrograma na oba načina 

In [ ]:
def lpFirFilter(fa, fp, Aa, Ap, fs):
    # Metod ogranicavanja impulsnoog odziva
    # nadjemo minimalni propusni opseg ovde je samo jedan 
    
    Wa = 2 * np.pi * fa / fs 
    Wp = 2 * np.pi*  fp / fs
    Wc = (Wa + Wp)/2
    Bt = Wa - Wp
    
    # Delta
    delta_p = (10**(0.05*Ap)-1)/(10**(0.05*Ap)+1)
    delta_a = 10**(-0.05*Aa)
    
    delta = min([delta_p, delta_a])

    # Alpha_a
    AaSpec = -20*np.log10(delta)
        
    # Beta
    beta = 0;
    if AaSpec < 21:
        beta = 0
    elif AaSpec <= 50:
        beta = 0.5842*(AaSpec-21)**0.4 + 0.07886*(AaSpec-21)
    else:
        beta = 0.1102*(AaSpec-8.7)
    
    # Filter order
    if AaSpec <= 21:
        D = 0.9222
    else:
        D = (AaSpec-7.95)/14.36
    M = int(np.ceil(2*np.pi*D/Bt + 1))
    N = M - 1
    numOfPoints = 10000
    filterFitsInSpecs = False
    while not filterFitsInSpecs:
        #odredimo kajzerovu prozorsku funkciju
        w = signal.kaiser(N + 1, beta)

        na = np.arange(-N / 2, N / 2 + 1, 1)
       
        # idealni odziv nakon sto ga shiftujemo za (M - 1) / 2
        n = np.arange(0, N + 1)
        hid = np.zeros(N + 1)
        if (N % 2 == 0):
            na = np.delete(na, N//2)
            hid[N // 2] = 1 - (Wc) / np.pi
            hid[n != N // 2] =  np.sin(Wc * na)  / (np.pi * na)
        else:
            hid[n] = np.sin(Wc * na)  / (np.pi * na)

        h = hid * w
        fd, Hd = signal.freqz(h, 1, worN=numOfPoints)
        Hda = np.abs(Hd)
    
        # izracunavanje tacaka dokle cemo proveravati
        deltaW = np.pi / numOfPoints
        kPB1 = int(np.ceil(Wp / deltaW))
        kSB1 = int(np.floor(Wa / deltaW))
        
        # provera da li su specifikaicje zadovoljene
        if np.all(Hda[0 : kPB1] < 1 + delta_p) and \
                np.all(Hda[0:kPB1] > 1 - delta_p) and \
                np.all(Hda[kSB1 : ] < delta_a) :
            filterFitsInSpecs = True;
        else:
            N += 1
    print("Red NF filtra za projektovanje metodom ogranicavanja impulsnog odziva: {}".format(N))
    return h

In [ ]:
def bsFirFilter(fa1, fa2, fp1, fp2, Aa ,Ap, fs):
    # Nadjemo minimalni od svih opsega
    Wa1, Wp1 = 2 * np.pi * fa1 / fs, 2 * np.pi * fp1 / fs
    Wa2, Wp2 = 2 * np.pi * fa2 / fs, 2 * np.pi * fp2 / fs
    B = min([abs(Wa1 - Wp1), abs(Wp2 - Wa2)])
    Wc1 = (Wa1 + Wp1) / 2
    Wc2 = (Wa2 + Wp2) / 2
    
    # Nadjemo minimalno delta
    delta_a = 10 ** (-0.05 * Aa)
    delta_p = (10 ** (0.05 * Ap) - 1) / (10 ** (0.05 * Ap) + 1)
    delta = min(delta_p, delta_a)
    
    # Izracunamo parametar B kajzerove prozorske funkcije 
    if delta < delta_a:
        AaSpec = -20 * np.log10(delta)
    else:
        AaSpec = Aa

    if AaSpec < 21:
        beta = 0
    elif AaSpec <= 50:
        beta = 0.5842 * (AaSpec - 21) ** 0.4 + 0.07886 * (AaSpec - 21)
    else:
        beta = 0.1102 * (AaSpec - 8.7)
    
    # Izracunamo parametar D koji je kasnije potreban za odredjivanje broja odbiraka
    if AaSpec <= 21:
        D = 0.9222
    else:
        D = (AaSpec-7.95)/14.36
    # Izracunavanje broja odbiraka filtra
    M = int(np.ceil(2 * np.pi * D / B + 1))
    N = M - 1
    numOfPoints = 10000
    filterFitsInSpecs = False

    while not filterFitsInSpecs:
       
        w = signal.kaiser(N + 1, beta)

        # sad se izracuna hID pa se shiftuje za (M-1) / 2 i nakon toga smo dobili trazeni filtar
        # idealni odziv koji dobijemo pre pomeranja, njegovi indeksi
        na = np.arange(-N / 2, N / 2 + 1, 1)
        # idealni odziv nakon sto ga shiftujemo za (M - 1) / 2
        n = np.arange(0, N + 1)
        hid = np.zeros(N + 1)
        if (N % 2 == 0):
            na = np.delete(na, N//2)
            hid[N // 2] = 1 - (Wc2 - Wc1) / np.pi
            hid[n != N // 2] = (np.sin(Wc1 * na) - np.sin(Wc2 * na)) / (np.pi * na)
        else:
            hid[n] = (np.sin(Wc1 * na) - np.sin(Wc2 * na)) / (np.pi * na)

        h = hid * w
        fd, Hd = signal.freqz(h, 1, worN=numOfPoints)
        Ha = np.abs(Hd)
        
        #odredjivanje tacaka dokle cemo proveravati
        DeltaW = np.pi / numOfPoints
        kSB1 = int(np.floor(Wa1/DeltaW))
        kPB1 = int(np.ceil(Wp1/DeltaW))
        kPB2 = int(np.floor(Wp2/DeltaW))
        kSB2 = int(np.ceil(Wa2/DeltaW))

        # provera da li su specifikacije zadovoljene u linearnoj razmeri

        if np.all(Ha[:kPB1] > 1 - delta_p) and np.all(Ha[:kPB1] < 1 + delta_p) and \
           np.all(Ha[kSB1:kSB2] < delta_a) and np.all(Ha[kPB2:] > 1 - delta_p) and \
            np.all(Ha[kPB2:] < 1 + delta_p):
            filterFitsInSpecs = True
        else:
      
            N += 1

    print("Red NO filtra za projektovanje metodom FIR:", N)
    return h

In [ ]:
def lpRemezFilter(fa, fp, fs, Aa, Ap):
    #1. nadjemo delta_p i delta_a
    delta_p = (10 ** (0.05 * Ap) - 1) / (10 ** (0.05 * Ap) + 1)
    delta_a = 10 ** (-0.05 * Aa)
    Wa = 2 * np.pi * fa / fs
    Wp = 2 * np.pi * fp / fs
    Wc = (Wa + Wp) / 2

    #2. izracunamo red filtra iz datih preporucenih formula
    # treba istaci da ovo nisu egzaktne formule, vec one koje najblize predvidjaju red filtra empirijski dobijene
    D = (0.005309 * np.log10(delta_p) ** 2 + 0.07114 * np.log10(delta_p) - 0.4761) * np.log10(delta_a)
    D = D - (0.00266 * np.log10(delta_p) ** 2 + 0.5941 * np.log10(delta_p) + 0.4278)
    f = 11.01217 + 0.51244 * (np.log10(delta_p) - np.log10(delta_a))
    Bt = abs((fp - fa)) / fs * 2 * np.pi
    M = int(np.ceil(2 * np.pi * D / Bt - f * Bt / (2 * np.pi) + 1))
    
    if M % 2 == 0:
        M += 1
    
    #3, filtar se dobija formulom signal.remez(...)
    Hid = [1, 0]
    F = [0, fp / fs, fa / fs, 1 / 2]
    W = [1, delta_p / delta_a]
    h  = signal.remez(M, F, Hid, W)

    #4. potrebno je proveriti da li zadati filtar zadovoljava specifikacije
    filterFitsInSpecs = False
    while not filterFitsInSpecs:
        h = signal.remez(M, F, Hid, W)

        numOfPoints = 10000
        deltaW =  np.pi / numOfPoints
        kPB1 = int(np.ceil(Wp / deltaW))
        kSB1 = int(np.floor(Wa / deltaW))
        fd, Hd = signal.freqz(h, 1, worN = numOfPoints)
        Hda = abs(Hd)
        if np.all(Hda[0: kPB1] < 1 + delta_p) and \
                np.all(Hda[0:kPB1] > 1 - delta_p) and \
                np.all(Hda[kSB1: ] < delta_a):
            filterFitsInSpecs = True;
        else:
            M += 1
    N = M - 1
    print("Red optimizacionim postupkom za NF filtar je {}".format(N))
    return h

In [ ]:
def bsRemezFilter(fs, fa1, fp1, fa2, fp2, Aa, Ap):
    # 1. nadjemo delta_p i delta_a
    delta_p = (10 ** (0.05 * Ap) - 1) / (10 ** (0.05 * Ap) + 1)
    delta_a = 10 ** (-0.05 * Aa)
    Wa1 = 2 * np.pi * fa1 / fs
    Wp1 = 2 * np.pi * fp1 / fs
    Wc1 = (Wa1 + Wp1) / 2
    Wa2 = 2 * np.pi * fa2 / fs
    Wp2 = 2 * np.pi * fp2 / fs
    Wc2 = (Wa2 + Wp2) / 2

    # 2. izracunamo red filtra iz datih preporucenih formula
    D = (0.001201 * np.log10(delta_p) ** 2 + 0.09664 * np.log10(delta_p) - 0.51325) * np.log10(delta_a)
    D = D + 0.00203 * np.log10(delta_p) ** 2 - 0.57054 * np.log10(delta_p) - 0.44314
    f = -16.9 - 14.6 * (np.log10(delta_p) - np.log10(delta_a))
    B1 = abs(Wa1 - Wp1)
    B2 = abs(Wp2 - Wa2)
    B = min([B1, B2])
    M = int(np.ceil(2 * np.pi * D / B - f * B / (2 * np.pi) + 1))

    #3. Odredimo ucestanosti i proverimo da li nam red filtra zadovoljava specifikacije 

    Fp1, Fp2, Fa1, Fa2 = fp1 / fs, fp2 / fs, fa1 / fs, fa2 / fs


    F = [0, Fp1, Fa1, Fa2, Fp2, 0.5]
    Hid = [1, 0 , 1]
    W = [1, delta_p / delta_a, 1]
    numOfPoints = 10000
    filterFitsInSpecs = False
    while not filterFitsInSpecs:
        h = signal.remez(M, F, Hid, W)
        fd, Hd = signal.freqz(h, 1, worN=numOfPoints)
        Ha = abs(Hd)
      
        DeltaW = np.pi / numOfPoints
        kSB1 = int(np.ceil(Wa1 / DeltaW))
        kPB1 = int(np.floor(Wp1 / DeltaW))
        kSB2 = int(np.ceil(Wa2 / DeltaW))
        kPB2 = int(np.floor(Wp2 / DeltaW))

        if np.all(Ha[:kPB1] > 1 - delta_p) and np.all(Ha[:kPB1] < 1 + delta_p) and \
            np.all(Ha[kSB1:kSB2] < delta_a) and np.all(Ha[kPB2:] > 1 - delta_p) and \
            np.all(Ha[kPB2:] < 1 + delta_p):
            filterFitsInSpecs = True
        else:
            M += 1
    N = M-1
    print("Red optimizacionim postupkom za NO filtar je {}".format(N))
    return h

## Filtriranje signala optimizacionim postupkom

In [ ]:
fs=48000
h1_remez = bsRemezFilter(fs=fs, fa1=1350, fp1=1250, fa2=1650, fp2=1750, Aa=40, Ap=0.2)
h2_remez = bsRemezFilter(fs=fs, fa1=2450, fp1=2350, fa2=2750, fp2=2850, Aa=40, Ap=0.2)
h3_remez = lpRemezFilter(fs=fs, fp=6000, fa=6500, Aa=40, Ap=0.2)

filtered_remez = signal.lfilter(h1_remez, 1, corrupted_signal)
filtered_remez1 = signal.lfilter(h2_remez, 1, filtered_remez)
filtered_remez2 = signal.lfilter(h3_remez, 1, filtered_remez1)

fig,ax = plt.subplots(figsize = (11,4))
plt.subplots_adjust(bottom=0.20)
fMaxShow = 8000

Nwin = 3*1024;
window = signal.hamming(Nwin) 
NFFT = Nwin

# Prikazivanje spektrograma
f, t, Sxx  = signal.spectrogram(filtered_remez2, fs = fs,  window = window, noverlap = Nwin // 8, nfft = NFFT, return_onesided=True, 
                               scaling='spectrum', mode='complex')

fMaxIndex = NFFT*fMaxShow//fs

ax.pcolormesh(t, f[:fMaxIndex], 20*np.log10(abs(Sxx[:][:fMaxIndex])), shading = 'nearest', cmap = "magma")# try shading='gouraud')
ax.set_xlabel(r"$t[s]$")
ax.set_ylabel(r"$f$ [Hz]")
ax.set_title("Spektrogram filtriranog signala")

## Filtriranje signala metodom ograničavanja impulsnog odziva 

In [ ]:
fs = 48000



# Poziv funkcije da dobijemo koeficijente
h1 = bsFirFilter(fs=fs, fa1=1350, fp1=1250, fa2=1650, fp2=1750, Aa=40, Ap=0.2)
h2 = bsFirFilter(fs=fs, fa1=2450, fp1=2350, fa2=2750, fp2=2850, Aa=40, Ap=0.2)
h3 = lpFirFilter(6500, 6000, Aa = 40, Ap = 0.2, fs = 48000)

# Filtriranje signala redom
filtered_1 = signal.lfilter(h1, 1, corrupted_signal)
filtered_2 = signal.lfilter(h2, 1, filtered_1)
filtered_3 = signal.lfilter(h3, 1, filtered_2)

fig,ax = plt.subplots(figsize = (11,4))
plt.subplots_adjust(bottom=0.20)
fMaxShow = 8000

Nwin = 3*1024;
window = signal.hamming(Nwin) 
NFFT = Nwin

# Prikazivanje spektrograma
f, t, Sxx  = signal.spectrogram(filtered_3, fs = fs,  window = window, noverlap = Nwin // 8, nfft = NFFT, return_onesided=True, 
                               scaling='spectrum', mode='complex')

fMaxIndex = NFFT*fMaxShow//fs
ax.pcolormesh(t, f[:fMaxIndex], 20*np.log10(abs(Sxx[:][:fMaxIndex])), shading = 'nearest', cmap = "magma")# try shading='gouraud')
ax.set_xlabel(r"$t[s]$")
ax.set_ylabel(r"$f$ [Hz]")
ax.set_title("Spektrogram filtriranog signala")

### **Zaključak**

Signal je uspešno očišćen sa oba metoda: optimizacionim i metodom ograničavanja impulsa. Kod optimizacionog metoda red NO filtara je  značajno manji nego red  kod projektovanja metodom ograničavanja impulsa( za nekih stotinak). Što se tiče projektovanja za NF filtar njegov red je manji sa projektovanjem optimizacionim postupkom nego sa projektovanjem metodom ograničavanja impulsa.Spektrogrami potvrđuju da su oba pristupa funkcionalna u uklanjanju neželjenih komponenti.

## 2.4 Učitavanje signala na zvucnicima 


### Učitavanje signala koji je isfiltriran metodom ograničavanja impulsnog odziva

In [ ]:
import IPython
from scipy.io import wavfile

fs = 48000



# Poziv funkcije da dobijemo koeficijente
h1 = bsFirFilter(fs=fs, fa1=1350, fp1=1250, fa2=1650, fp2=1750, Aa=40, Ap=0.2)
h2 = bsFirFilter(fs=fs, fa1=2450, fp1=2350, fa2=2750, fp2=2850, Aa=40, Ap=0.2)
h3 = lpFirFilter(6500, 6000, Aa = 40, Ap = 0.2, fs = 48000)

# Filtriranje signala redom
filtered_1 = signal.lfilter(h1, 1, corrupted_signal)
filtered_2 = signal.lfilter(h2, 1, filtered_1)
filtered_3 = signal.lfilter(h3, 1, filtered_2)

# Ucitavanje signala na zvucnicima
IPython.display.display(IPython.display.Audio(filtered_3, rate = fs))

### Učitavanje signala koji je isfiltriran optimizacionim postupkom 

In [ ]:
fs=48000

# Poziv funkcije da dobijemo koeficijente
h1_remez = bsRemezFilter(fs=fs, fa1=1350, fp1=1250, fa2=1650, fp2=1750, Aa=40, Ap=0.2)
h2_remez = bsRemezFilter(fs=fs, fa1=2450, fp1=2350, fa2=2750, fp2=2850, Aa=40, Ap=0.2)
h3_remez = lpRemezFilter(fs=fs, fp=6000, fa=6500, Aa=40, Ap=0.2)

# Filtriranje signala redom
filtered_remez = signal.lfilter(h1_remez, 1, corrupted_signal)
filtered_remez1 = signal.lfilter(h2_remez, 1, filtered_remez)
filtered_remez2 = signal.lfilter(h3_remez, 1, filtered_remez1)

# Ucitavanje signala na zvucnicima
IPython.display.display(IPython.display.Audio(filtered_3, rate = fs))

## Pomoćne funkcije za 2.5 tačku

In [ ]:
def plotNOFilterResponse(h, fs, fp1, fa1, fa2, fp2, Ap, Aa):
    Wd, Hd = signal.freqz(h, 1, worN=10000)
    
    # Konverzija granica u digitalnu frekvenciju 
    Wp1 = 2 * np.pi * fp1 / fs
    Wa1 = 2 * np.pi * fa1 / fs
    Wa2 = 2 * np.pi * fa2 / fs
    Wp2 = 2 * np.pi * fp2 / fs
    
    fig, axs = plt.subplots(1, 2, figsize=(12, 5))
    plt.subplots_adjust(bottom=0.25, wspace=0.4)
    
    # Amplitudska karakteristika
    ax1 = axs[0]
    ax1.plot(Wd, 20 * np.log10(np.abs(Hd)))
    ax1.set_title("Amplitudska karakteristika")
    ax1.set_xlabel(r"$\Omega$ [rad]")
    ax1.set_ylabel(r"$|H(e^{j\Omega})|$ [dB]")
    ax1.set_xlim([0, np.pi/6])
    ax1.set_ylim([-110, 10])
    ax1.grid(True)
    
    # Prvi propusni opseg
    ax1.add_patch(mpl.patches.Polygon([[0, -Ap - 3], [Wp1, -Ap - 3], [Wp1, -Ap], [0, -Ap]], closed=True, fill=False, hatch='////', color='red'))
    ax1.add_patch(mpl.patches.Polygon([[0, Ap], [Wp1, Ap], [Wp1, Ap + 3], [0, Ap + 3]], closed=True, fill=False, hatch='////', color='red'))
    
    # Nepropusni opseg
    ax1.add_patch(mpl.patches.Polygon([[Wa1, -Aa + 3], [Wa2, -Aa + 3], [Wa2, -Aa], [Wa1, -Aa]], closed=True, fill=False, hatch='////', color='red'))
    
    # Drugi propusni opseg
    ax1.add_patch(mpl.patches.Polygon([[Wp2, -Ap - 3], [np.pi, -Ap - 3], [np.pi, -Ap], [Wp2, -Ap]], closed=True, fill=False, hatch='////', color='red'))
    ax1.add_patch(mpl.patches.Polygon([[Wp2, Ap], [np.pi, Ap], [np.pi, Ap + 3], [Wp2, Ap + 3]], closed=True, fill=False, hatch='////', color='red'))
    
    # Fazna karakteristika
    ax2 = axs[1]
    ax2.plot(Wd, np.unwrap(np.angle(Hd)))
    ax2.set_title("Fazna karakteristika")
    ax2.set_xlabel(r"$\Omega$ [rad]")
    ax2.set_ylabel(r"$\angle H(e^{j\Omega})$ [rad]")
    ax2.set_xlim([0, np.pi])
    ax2.grid(True)
    
    plt.show()

In [ ]:
def plotLPFilterResponse(h, fs, fp, fa, Ap, Aa):
    Wd, Hd = signal.freqz(h, 1, worN = 10000)
    
    # Konverzija granica u digitalnu frekvenciju 
    Wp = 2 * np.pi * fp / fs
    Wa = 2 * np.pi * fa / fs
    
    fig, axs = plt.subplots(1, 2, figsize=(12, 5))
    plt.subplots_adjust(bottom=0.25, wspace=0.4)
    
    # Amplitudska karakteristika
    ax1 = axs[0]
    ax1.plot(Wd, 20 * np.log10(np.abs(Hd)))
    ax1.set_title("Amplitudska karakteristika")
    ax1.set_xlabel(r"$\Omega$ [rad]")
    ax1.set_ylabel(r"$|H(e^{j\Omega})|$ [dB]")
    ax1.set_xlim([0, np.pi ])
    ax1.set_ylim([-125, 10])
    ax1.grid(True)
    
    # Propusni opseg
    ax1.add_patch(mpl.patches.Polygon([[0, -Ap - 3], [Wp, -Ap - 3], [Wp, -Ap], [0, -Ap]], closed=True, fill=False, hatch='////', color='red'))
    ax1.add_patch(mpl.patches.Polygon([[0, Ap], [Wp, Ap], [Wp, Ap + 3], [0, Ap + 3]], closed=True, fill=False, hatch='////', color='red'))
    
    # Nepropusni opseg
    ax1.add_patch(mpl.patches.Polygon([[Wa, -Aa + 3], [np.pi, -Aa + 3], [np.pi, -Aa], [Wa, -Aa]], closed=True, fill=False, hatch='////', color='red'))
    
    
    # Fazna karakteristika
    ax2 = axs[1]
    ax2.plot(Wd, np.unwrap(np.angle(Hd)))
    ax2.set_title("Fazna karakteristika")
    ax2.set_xlabel(r"$\Omega$ [rad]")
    ax2.set_ylabel(r"$\angle H(e^{j\Omega})$ [rad]")
    ax2.set_xlim([0, np.pi])
    ax2.grid(True)
    
    plt.show()

## 2.5 Amplitudske i fazne karakteristike filtara 


### Filtri projektovani optimizacionim postupkom karakteristike 

NO filtar na 1500 Hz projektovan optimizacionim postupkom: 

In [ ]:
np, mpl, plt, signal = importEssentialLibs(True)


fs = 48000
h1_remez = bsRemezFilter(fs = fs, fa1 = 1350, fp1 = 1250, fa2 = 1650, fp2 = 1750, Aa = 40, Ap = 0.2)
plotNOFilterResponse(h1_remez, fs = 48000, fp1 = 1250, fa1 = 1350, fa2 = 1650, fp2 = 1750, Aa = 40, Ap = 0.2)

NO filtar na 2700 Hz projektovan optimizacionim postupkom:


In [ ]:
fs = 48000

h2_remez = bsRemezFilter(fs = fs, fa1 = 2450, fp1 = 2350, fa2 = 2750, fp2 = 2850, Aa = 40, Ap = 0.2)
fd2,Hd2_remez = signal.freqz(h2_remez, 1, worN = 10000)
plotNOFilterResponse(h2_remez, fs = 48000, fp1 = 2350, fa1 = 2450, fa2 = 2750, fp2 = 2850, Aa = 40, Ap = 0.2)

NF filtar na 6500+ Hz projektovan optimizacionim postupkom:

In [ ]:
fs = 48000

h3_remez = lpRemezFilter(fs=fs, fp=6000, fa=6500, Aa=40, Ap=0.2)
plotLPFilterResponse(h3_remez,fs, fp=6000, fa=6500, Aa=40, Ap=0.2)

## Filtri projektovani ograničavanjem impulsnog odziva

NO filtar na 1500 Hz projektovan ograničavanjem impulsnog odziva: 

In [ ]:
fs = 48000
h1 = bsFirFilter(fs = fs, fa1 = 1350, fp1 = 1250, fa2 = 1650, fp2 = 1750, Aa = 40, Ap = 0.2)
plotNOFilterResponse(h1, fs = 48000, fp1 = 1250, fa1=1350, fa2 = 1650, fp2 = 1750, Aa = 40, Ap = 0.2)

NO filtar na 2700 Hz projektovan ograničavanjem impulsnog odziva: 

In [ ]:
fs = 48000
h2 = bsFirFilter(fs = fs, fa1 = 2450, fp1 = 2350, fa2 = 2750, fp2 = 2850, Aa = 40, Ap = 0.2)
plotNOFilterResponse(h2, fs = 48000, fp1 = 2350, fa1 = 2450, fa2 = 2750, fp2 = 2850, Aa = 40, Ap = 0.2)

NF filtar na 6500+ Hz projektovan ograničavanjem impulsnog odziva: 

In [ ]:
fs = 48000
h3 = lpFirFilter(6500, 6000, Aa = 40, Ap = 0.2, fs = 48000)
plotLPFilterResponse(h3, fs, fp=6000, fa=6500, Aa=40, Ap=0.2)

# DEO 3

## 3.1 Implementacija funkcije sa float-point tačnošću

In [ ]:
def iirDirectCan(b, a, x):
    y = np.zeros(len(x))
    N, M = len(a), len(b)
    # y[n] = b_0 x[n] + b_1 x[n-1] + ... + b_M x[n-M] - (a_1 y[n-1] + a_2 y[n-2] + ... + a_N y[n-N])
    # y[0]=  b0 x[0],
    # y[1] = b1x[0] + box[1] - a1y[0],
    # y[2] = b2x[0] + b1x[1] + b2x[2] - a2y[0] - a1y[1]
    # prvi x i zadnji b se mnoze i prvi y i zadnji a se mnoze
    for n in range(len(x)):
        #izracunam koeficijente za b, moram da M da uzmem da bi dobio vrednost izraza
        for i in range(M):
            if(n-i>= 0):
                y[n] += b[i] * x[n-i]

        # ako je i a0 prosledjeno racunam bez njega
        for j in range(1, N):
            if(n-j >= 0):
                y[n] -= a[j] * y[n-j]

    return y

## 3.2 Implementacija funkcije sa fixed-point tačnošću 

In [ ]:
def iirDirectCanFΧP(b, a, x, outFxpFormat):
    y = np.zeros(len(x))
    N, M = len(a), len(b)

  
    y = Fxp(x, dtype = outFxpFormat)
    y.overflow = x.overflow
    y.rounding = x.rounding

    for n in range(len(x)):
        #izracunam koeficijente za b, moram da M i N da zauzmem da bi dobio vrednost izraza
        in_between_sum1 = Fxp(0, like = b[0] * x[0])
        for i in range(M):
            if(n-i>= 0):
                in_between_sum1 += b[i] * x[n-i]
        in_between_sum2 = Fxp(0, like = a[1] * y[0])
        
        # ako je i a0 prodledjeno racunam bez njega
        for j in range(1, N):
            if(n-j >= 0):
                in_between_sum2 -= a[j] * y[n-j]
        y[n] = Fxp(in_between_sum1 + in_between_sum2, like = y)
    return y

## 3.3 Provera prve funkcije sa iir filtrom PowerLineNoise(...) iz tačke 1 

In [ ]:
np, mp, plt, signal = importEssentialLibs(True) 
import scipy.fft as fft 

fs = 360
fc = 60
Aa = 40
Ap = 0.5


h = powerLineNoiseFilter(fs, fc, Aa, Ap)
b, a = h[0], h[1] 

N = 360
n = np.arange(N)

# Normalizovane frekvencije (frekvencija / fs)
F1 = 10 / fs # korisna
F2 = 20 / fs   # korisna
F3 = 60 / fs   # neželjena, šum koji treba ukloniti

# Zbir tri sinusoide: 10 Hz, 20 Hz i 60 Hz (koja treba da se ukloni)
x = np.sin(2 * np.pi * F1 * n) + np.sin(2 * np.pi * F2 * n) + 0.7 * np.sin(2 * np.pi * F3 * n)
X = fft.fft(x)
k = np.arange(N) - (N//2)
fx = fs * k / N 

fig = plt.figure(figsize=(10, 4))
plt.subplots_adjust(bottom=0.25, wspace=0.4)

#Skiciranje x
ax1 = fig.add_subplot(1, 2, 1)
ax1.stem(fx, abs(fft.fftshift(X)))
ax1.set_xlabel(r'$f_k$')
ax1.set_ylabel(r'$|X[f_k]|$')

# Skiciranje y
y = iirDirectCan(b, a, x)
Y = fft.fft(y)
fy = fx  

ax2 = fig.add_subplot(1, 2, 2)
ax2.stem(fy, abs(fft.fftshift(Y)))
ax2.set_xlabel(r'$f_k$')
ax2.set_ylabel(r'$|Y[f_k]|$')


plt.show()

## 3.4.1 Određivanje broja koeficijenata da bi filtar ostao stabilan 

In [ ]:
np, mp, plt, signal = importEssentialLibs(True)
from fxpmath import Fxp
fs = 360
fc = 60
Aa = 40
Ap = 0.5


h = powerLineNoiseFilter(fs, fc, Aa, Ap)
b, a = h[0], h[1]

# overflow moze bit 'saturate' ili 'wrap'
overFlowMethod = 'saturate'
# rounding moze biti 'floor', 'trunc', 'around', 'ceil', 'fix'
roundingMethod = 'around'
fig = plt.figure(figsize=(12,5))
plt.subplots_adjust(bottom = 0.15,wspace = 0.5)

ax1 = fig.add_subplot(1,2,1)
# n_word[b,a] = 14,15 , n_frac = [7,8] sistem je nestabilan blizu stabilnog 
bFxp = Fxp(b, signed = True, n_word = 10, n_frac = 7, overflow = overFlowMethod, rounding = roundingMethod)
aFxp = Fxp(a, signed = True, n_word = 11, n_frac = 8, overflow = overFlowMethod, rounding = roundingMethod)

b_fx1 = bFxp.get_val()
a_fx1 = aFxp.get_val()
ax1.set_title("Nestabilan Filtar")
zplane(b_fx1, a_fx1, ax1)

# na nword[b, a] = 17, i nfrac[b,a] = 13 je tu oko minimalne vrednosti da filtar i dalje ostane stabilan 
bFxp2 = Fxp(b, signed = True, n_word = 17, n_frac = 13, overflow = overFlowMethod, rounding = roundingMethod)
aFxp2 = Fxp(a, signed = True, n_word = 17, n_frac = 13, overflow = overFlowMethod, rounding = roundingMethod)

b_fx2 = bFxp2.get_val()
a_fx2 = aFxp2.get_val()


ax2 = fig.add_subplot(1,2,2)
ax2.set_title("Stabilan Filtar")
zplane(b_fx2, a_fx2, ax2)

### **Zaključak**
Filtar je i dalje stabilan za n_word = 17 ( dužina celobrojnog dela je 17) i za n_frac = 13 (dužina razlomljenog dela je 13).Ovo bi bile minimalne vrednosti za koje je filtar stabilan.

## 3.4.2 Određivanje koeficijenata tako da amplitudska karakteristika ne odstupa previše

In [ ]:
np, mp, plt, signal = importEssentialLibs(True)
from fxpmath import Fxp
import ipywidgets as widgets

def minAmplitudeSpectar(n_word,n_frac):
    fs = 360
    fc = 60
    Aa = 40
    Ap = 0.5

    h = powerLineNoiseFilter(fs, fc, Aa, Ap)
    b, a = h[0], h[1]

    fd, Hd = signal.freqz(b, a , worN = 1024)

    fig, ax = plt.subplots(figsize=(10,5))
    plt.subplots_adjust(bottom = 0.15)

    ax.plot(fd, 20 * np.log10(np.abs(Hd)), label="Originalni filter")

    bFxp = Fxp(b, signed = True, n_word = n_word, n_frac = n_frac, overflow = overFlowMethod, rounding = roundingMethod)
    aFxp = Fxp(a, signed = True, n_word = n_word, n_frac = n_frac, overflow = overFlowMethod, rounding = roundingMethod)

    b_fx = bFxp.get_val()
    a_fx = aFxp.get_val()
    fd_fx, Hd_fx = signal.freqz(b_fx, a_fx, worN=1024)

    ax.plot(fd_fx, 20 * np.log10(np.abs(Hd_fx)), label="Filter sa kvantizacijom")

    ax.set_xlabel(r"$\Omega$")
    ax.set_ylabel(r"$|H(\Omega)| [dB]$")
    ax.set_xlim([0.5,2])
    ax.grid(True)
    ax.legend()

    plt.show()
widgets.interact(minAmplitudeSpectar, n_word = (1, 100, 1), n_frac = (1, 100, 1) )

### Iz widgeta se da zaklučiti da su minimalne širine bita za koje se amplitudske karakteristike u potpunosti poklapaju sledeće: **n_frac = 22** i **n_word = 26**.

## 3.5 Uporedjivanje rezultata 

In [ ]:
np, mp, plt, signal = importEssentialLibs(True) 
import scipy.fft as fft 
def plotGraphs(outFxpFormat):

    fs = 360
    fc = 60
    Aa = 40
    Ap = 0.5


    h = powerLineNoiseFilter(fs, fc, Aa, Ap)
    b, a = h[0], h[1] 

    N = 360
    n = np.arange(N)

    # Normalizovane frekvencije (frekvencija / fs)
    F1 = 10 / fs # korisna
    F2 = 20 / fs   # korisna
    F3 = 60 / fs   # neželjena, šum koji treba ukloniti


    # overflow moze bit 'saturate' ili 'wrap'
    overFlowMethod = 'saturate'
    # rounding moze biti 'floor', 'trunc', 'around', 'ceil', 'fix'
    roundingMethod = 'around'

    # Zbir tri sinusoide: 10 Hz, 20 Hz i 60 Hz (koja treba da se ukloni)
    x = np.sin(2 * np.pi * F1 * n) + np.sin(2 * np.pi * F2 * n) + 0.7 * np.sin(2 * np.pi * F3 * n)
    xFxp = Fxp(x, signed = True, n_word = 19, n_frac = 16, overflow = overFlowMethod, rounding = roundingMethod)
    bFxp = Fxp(b, signed = True, n_word = 26, n_frac = 22, overflow = overFlowMethod, rounding = roundingMethod)
    aFxp = Fxp(a, signed = True, n_word = 26, n_frac = 22, overflow = overFlowMethod, rounding = roundingMethod)


    #skiciranje figure 
    fig = plt.figure(figsize=(12,5))
    plt.subplots_adjust(bottom = 0.15,wspace = 0.5,hspace = 0.7)

    ax1 = fig.add_subplot(2, 2, 1)  # 2 reda, 2 kolone, 1. pozicija
    ax1.plot(n, x)
    ax1.set_title("Originalni signal") 
    ax1.set_xlabel(r"$n$")
    ax1.set_ylabel(r"$x[n]$")

    ax2 = fig.add_subplot(2, 2, 2)  # 2 reda, 2 kolone, 2. pozicija
    y = iirDirectCan(b, a, x)
    ax2.plot(n, y)
    ax2.set_title("Signal filtriran sa float preciznošću") 
    ax2.set_xlabel(r"$n$")
    ax2.set_ylabel(r"$y[n]$")

    ax3 = fig.add_subplot(2, 2, 3)  # 2 reda, 2 kolone, 3. pozicija
    yFxp =  iirDirectCanFΧP(bFxp, aFxp, xFxp, outFxpFormat)
    ax3.plot(n, yFxp)
    ax3.set_title("Signal filtriran sa fixed preciznošću") 
    ax3.set_xlabel(r"$n$")
    ax3.set_ylabel(r"$yFxp[n]$")
    
    ax4 = fig.add_subplot(2, 2, 4)  # 2 reda, 2 kolone, 4. pozicija
    ax4.plot(n, yFxp - y)
    ax4.set_xlabel(r'$n$')
    ax4.set_ylabel(r'$e[n]$');
    ax4.set_title(r"$y_{Fxp} - y$")
    plt.show()

In [ ]:
plotGraphs("fxp-s26/22")

## Pozivanje funkcije još 3 puta za razlicite Fxp formate 

### Prvi poziv 

In [ ]:
plotGraphs("fxp-s19/16")

### Drugi poziv 

In [ ]:
plotGraphs("fxp-s22/18")

### Treći Poziv

In [ ]:
plotGraphs("fxp-s16/14")

## 3.6 Zaključak na osnovu dijagrama:
1. **Što je veći broj frakcionih bitova (`n_frac`), to je greška manja.**  
   `s26/22` ima najmanju grešku jer koristi najviše bitova za preciznost.

2. **Manji formati (kao `s16/14`) uvode znatno veće greške.**  

3. **Greška ne zavisi samo od ukupnog broja bitova (`n_word`), već od odnosa  broj_integer_bita/broj_frakcionih_bita .**  
   Primer:`s22/18` ima manju grešku od `s19/16`, iako ima samo 3 bita više  za celu reč — jer ima 2 bita više za frakciju.